# Rock · Paper · Scissors with YOLO11 — workshop notebook 🪨 📄 ✂️

This notebook walks through the four stages of the computer-vision project that
powers our Rock-Paper-Scissors game:

1. **Dataset** — look at the images and labels the model learns from
2. **Training** — run a real (short!) training and watch the model improve
3. **Evaluation** — measure how much better the trained model got
4. **Inference** — use the model to detect hands, and even play a round

The same code lives in the standalone scripts (`prepare_dataset.py`, `train.py`,
`evaluate.py`, `play.py`) — the notebook just lets us run it piece by piece and
look at everything along the way.

**How to run this notebook** (from the `rps/` folder):

```bash
uv sync                                           # once: install the project deps
uv run --with jupyter jupyter lab workshop.ipynb
```

> Prerequisites: the dataset must be unpacked in `dataset/` (run
> `uv run prepare_dataset.py` once if it isn't), and for the comparisons in
> §3–§4 the fully-trained model should be at `runs/detect/rps_train/weights/best.pt`.

In [ ]:
import csv
import random
import re
import shutil
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import patches
from PIL import Image
from ultralytics import YOLO

import config       # the project's shared settings — read config.py first!
import game_logic   # the pure rules of the game (no ML in there)

DEVICE = config.get_device()

# One fixed colour per class, used in every figure of this notebook.
CLASS_COLORS = {"Paper": "#2a78d6", "Rock": "#e34948", "Scissors": "#eda100"}

# The fully-trained model (60 epochs) that the game actually plays with.
PRETRAINED_RUN = config.RUNS_DIR / "detect" / "rps_train"

# data.yaml stores an absolute path, so if this folder was moved or copied onto
# another machine, point it back at *our* dataset (and drop the stale caches).
text = config.DATA_YAML.read_text()
fixed = re.sub(r"(?m)^path: .*$", f"path: {config.DATASET_DIR}", text)
if fixed != text:
    config.DATA_YAML.write_text(fixed)
    for cache in config.DATASET_DIR.glob("*/labels.cache"):
        cache.unlink()
    print(f"data.yaml updated to point at {config.DATASET_DIR}")

print(f"Compute device : {DEVICE}")
print(f"Classes        : {config.CLASS_NAMES}")
print(f"Dataset ready  : {config.DATA_YAML.exists()}")
print(f"Trained model  : {(PRETRAINED_RUN / 'weights' / 'best.pt').exists()}")

## 1. Dataset

The model learns from a public [Roboflow dataset](https://universe.roboflow.com/roboflow-58fyf/rock-paper-scissors-sxsw)
of photos of hands playing rock, paper and scissors, split three ways:

| split | used for |
|---|---|
| **train** | the images the model learns from |
| **valid** | checked after every epoch, to see how training is going |
| **test**  | kept aside until the very end, for the final honest score |

Every image comes with a small text file listing what is in it: one line per
hand, with a **class id** (`0` Paper, `1` Rock, `2` Scissors) and a **bounding
box**. Quite a few images contain *no* hands at all — these "background" images
teach the model that sometimes there is simply nothing to detect.

In the table below, note that `hands` counts **labelled boxes**, not images:
every image is either *with hands* or *background*, but an image with hands can
contain more than one (e.g. both players in frame) — so
`images = with hands + background`, while `hands` can exceed `with hands`.

In [ ]:
def label_file(image_path: Path) -> Path:
    """The label that belongs to an image: same name, .txt, in ../labels/."""
    return image_path.parent.parent / "labels" / (image_path.stem + ".txt")


def load_boxes(image_path: Path) -> list[tuple[int, float, float, float, float]]:
    """Read a YOLO label file: one `class cx cy w h` line per object (all 0-1)."""
    lf = label_file(image_path)
    if not lf.exists():
        return []
    return [
        (int(c), float(x), float(y), float(w), float(h))
        for c, x, y, w, h in
        (line.split() for line in lf.read_text().splitlines() if line.strip())
    ]


train_class_counts = Counter()
print(f"{'split':8s} {'images':>7s} {'with hands':>11s} {'background':>11s} {'hands':>7s}")
for split in ("train", "valid", "test"):
    images = sorted((config.DATASET_DIR / split / "images").glob("*.jpg"))
    boxes_per_image = [load_boxes(p) for p in images]
    with_hands = sum(1 for b in boxes_per_image if b)
    background = len(images) - with_hands
    hands = sum(len(b) for b in boxes_per_image)
    if split == "train":
        for boxes in boxes_per_image:
            for cls, *_ in boxes:
                train_class_counts[config.CLASS_NAMES[cls]] += 1
    print(f"{split:8s} {len(images):7d} {with_hands:11d} {background:11d} {hands:7d}")

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(config.CLASS_NAMES,
              [train_class_counts[n] for n in config.CLASS_NAMES],
              color=[CLASS_COLORS[n] for n in config.CLASS_NAMES], width=0.6)
ax.bar_label(bars, padding=3)
ax.set_title("Labelled hands per class — train split")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Note the imbalance: there are ~40% more Rock examples than Paper or Scissors.
Keep that in mind for the per-class scores later.

Now let's *look* at the data — always the first thing to do on any ML project.
Each image is drawn with its **ground-truth** boxes (the human-made labels, not
model predictions). Re-run the cell with a different `seed` for a new batch.

In [ ]:
def show_labeled_samples(split: str = "train", n: int = 8, seed: int = 0) -> None:
    """Show `n` random images of a split with their ground-truth boxes drawn on."""
    images = sorted((config.DATASET_DIR / split / "images").glob("*.jpg"))
    labeled = [p for p in images if load_boxes(p)]   # skip background images here
    picks = random.Random(seed).sample(labeled, n)

    cols, rows = 4, (n + 3) // 4
    fig, axes = plt.subplots(rows, cols, figsize=(3.6 * cols, 3.6 * rows))
    for ax, img_path in zip(axes.flat, picks):
        img = Image.open(img_path)
        W, H = img.size
        ax.imshow(img)
        for cls, cx, cy, w, h in load_boxes(img_path):
            name = config.CLASS_NAMES[cls]
            x0, y0 = (cx - w / 2) * W, (cy - h / 2) * H   # centre+size -> corner
            ax.add_patch(patches.Rectangle((x0, y0), w * W, h * H, fill=False,
                                           edgecolor=CLASS_COLORS[name], linewidth=2))
            ax.text(x0, y0 - 4, name, color="white", fontsize=9,
                    bbox=dict(facecolor=CLASS_COLORS[name], edgecolor="none", pad=1.5))
        ax.axis("off")
    for ax in axes.flat[len(picks):]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


show_labeled_samples("train", n=8, seed=0)   # try: split="test", seed=42, ...

And this is what a label file actually looks like — the whole "ground truth" for
one image is just a few numbers in a text file:

In [ ]:
example = next(p for p in sorted((config.DATASET_DIR / "train" / "images").glob("*.jpg"))
               if len(load_boxes(p)) >= 2)
print(f"image : {example.name}")
print(f"label : {label_file(example).name}")
print()
print(label_file(example).read_text())
print("Each line is one hand:   class_id   centre_x   centre_y   width   height")
print("(coordinates are fractions of the image size -> they work at any resolution)")

## 2. Training

We do **transfer learning**: we start from `yolo11n.pt` — a small YOLO11 model
that has already learned a lot about everyday images — and teach it our three
hand shapes. Each full pass over the training images is called an **epoch**.
After every batch of images, the model compares its guesses with the labels,
measures how wrong it was (the **loss**), and nudges its internal numbers to do
slightly better next time. That's all "learning" is.

The model the game uses was trained for **60 epochs** (a couple of hours on a
MacBook). We don't have that kind of time — but we don't need it to *watch
learning happen*: even **3 epochs (~10 minutes on an Apple-Silicon Mac)** moves
the metrics visibly. Two knobs to play with:

* `EPOCHS` — how many passes over the data to run
* `FRACTION` — how much of the training set to use; `0.25` makes an epoch ~4×
  faster, at the price of learning a bit less per epoch

We also pass `save_period=1` so a snapshot of the model is kept after **every**
epoch — in §3 we'll use those snapshots to compare "1 epoch of learning" with
the final model.

While the cell runs, watch the log — the three losses shrink and `mAP50`
climbs, epoch after epoch:

| number | what it measures |
|---|---|
| `box_loss` | how far off the box **geometry** is — does the predicted rectangle overlap the labelled one, with the right centre and shape? |
| `cls_loss` | did the box get the right **name** (Rock vs Paper vs Scissors), confidently? |
| `dfl_loss` | how **sharply** the model pins down each box edge — the fine-tuning knob for localisation (the model predicts a small probability distribution per edge, and this rewards it for concentrating on the true position) |
| `mAP50` | **accuracy**, 0–1: a detection counts as correct if the class is right and the box overlaps the truth by ≥ 50%; per-class precision is averaged over all confidence thresholds, then over the classes |

One subtlety: the losses are measured on the **training** batches — that is
what the model directly optimises — while `mAP50` is measured on the
**validation** set after each epoch, which is what we actually care about.

In [ ]:
EPOCHS = 3        # a real training uses 60 (config.EPOCHS)
FRACTION = 1.0    # set to 0.25 for a ~4x faster demo

WORKSHOP_RUN = config.RUNS_DIR / "detect" / "rps_workshop"
if WORKSHOP_RUN.exists():
    shutil.rmtree(WORKSHOP_RUN)          # start fresh each time this cell runs

model = YOLO(config.BASE_MODEL)          # the pretrained starting point
model.train(
    data=str(config.DATA_YAML),          # where the images and labels are
    epochs=EPOCHS,
    fraction=FRACTION,
    imgsz=config.IMG_SIZE,               # 640 px, like the dataset
    batch=16,                            # images per learning step
    device=DEVICE,                       # the Mac's GPU ("mps")
    save_period=1,                       # snapshot the model after every epoch
    project=str(config.RUNS_DIR / "detect"),
    name="rps_workshop",
    exist_ok=True,
)

print("\nWorkshop model saved under:", WORKSHOP_RUN / "weights")

Training wrote a `results.csv` with one row per epoch. Let's plot it: loss
should go **down**, accuracy (mAP50) **up** — the model is learning. With only a
few epochs the curves are short, but the *direction* is what matters.

In [ ]:
def read_results(run_dir: Path) -> dict[str, list[float]]:
    """Load Ultralytics' results.csv as {column name -> list of values}."""
    with open(run_dir / "results.csv") as f:
        rows = list(csv.DictReader(f))
    return {key: [float(row[key]) for row in rows] for key in rows[0]}


def plot_training_curves(run_dir: Path, title: str) -> None:
    res = read_results(run_dir)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.suptitle(title)

    ax1.plot(res["epoch"], res["train/cls_loss"], marker="o", color="#2a78d6", label="train")
    ax1.plot(res["epoch"], res["val/cls_loss"], marker="o", color="#e34948", label="validation")
    ax1.set_title("classification loss — lower is better", fontsize=10)
    ax1.legend(frameon=False)

    ax2.plot(res["epoch"], res["metrics/mAP50(B)"], marker="o", color="#2a78d6", label="mAP50")
    ax2.plot(res["epoch"], res["metrics/mAP50-95(B)"], marker="o", color="#1baf7a", label="mAP50-95")
    ax2.set_title("accuracy on the validation set — higher is better", fontsize=10)
    ax2.set_ylim(0, 1)
    ax2.legend(frameon=False)

    for ax in (ax1, ax2):
        ax.set_xlabel("epoch")
        ax.spines[["top", "right"]].set_visible(False)
    plt.show()


plot_training_curves(WORKSHOP_RUN, f"Workshop training ({EPOCHS} epochs)")

For comparison, here is the **full 60-epoch training** of the model the game
plays with (that run ships with the project). Same picture, just carried on:
fast progress at the start, then the curves flatten out — most of the learning
happens early, and the later epochs only polish.

In [ ]:
plot_training_curves(PRETRAINED_RUN, "Full training (60 epochs)")

## 3. Evaluation

Training printed metrics on the **validation** set; for the final verdict we
score on the **test** split — 304 images the model has *never* seen, not even
for checking. The metrics, in plain language:

* **precision** — of the hands the model flagged, how many were correct?
* **recall** — of the hands that were really there, how many did it find?
* **mAP50** — overall accuracy (0–1) with a lenient box-overlap rule; the good headline number
* **mAP50-95** — the same idea with much stricter overlap rules; always lower

Thanks to `save_period=1`, we still have the model exactly as it was after **one
single epoch**. Let's line it up against our short training's best model — and
against the fully-trained one:

In [ ]:
epoch_ckpts = sorted(WORKSHOP_RUN.glob("weights/epoch*.pt"),
                     key=lambda p: int(re.search(r"\d+", p.stem).group()))
assert epoch_ckpts, "No snapshots found — run the training cell in §2 first."

candidates = {
    "1 epoch": epoch_ckpts[0],                               # snapshot after epoch 1
    f"{EPOCHS} epochs": WORKSHOP_RUN / "weights" / "best.pt",  # our short training
}
if (PRETRAINED_RUN / "weights" / "best.pt").exists():
    candidates["60 epochs"] = PRETRAINED_RUN / "weights" / "best.pt"


def score_on_test(weights: Path) -> dict[str, float]:
    """Run the model over the test split and return the four headline metrics."""
    m = YOLO(str(weights)).val(data=str(config.DATA_YAML), split="test",
                               device=DEVICE, plots=False, verbose=False,
                               project=str(config.RUNS_DIR / "detect"),
                               name="rps_workshop_eval", exist_ok=True)
    return {"mAP50": m.box.map50, "mAP50-95": m.box.map,
            "precision": m.box.mp, "recall": m.box.mr}


scores = {label: score_on_test(weights) for label, weights in candidates.items()}

metric_names = ["mAP50", "mAP50-95", "precision", "recall"]
print(f"{'model':12s}" + "".join(f"{m:>11s}" for m in metric_names))
for label, s in scores.items():
    print(f"{label:12s}" + "".join(f"{s[m]:11.3f}" for m in metric_names))

In [ ]:
labels = list(scores)
x = range(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(6.5, 3.5))
b1 = ax.bar([i - w / 2 for i in x], [scores[l]["mAP50"] for l in labels],
            w, color="#2a78d6", label="mAP50")
b2 = ax.bar([i + w / 2 for i in x], [scores[l]["mAP50-95"] for l in labels],
            w, color="#1baf7a", label="mAP50-95")
ax.bar_label(b1, fmt="%.2f", padding=2, fontsize=8)
ax.bar_label(b2, fmt="%.2f", padding=2, fontsize=8)
ax.set_xticks(list(x), labels)
ax.set_ylim(0, 1)
ax.set_title("Accuracy on the test split")
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Numbers are convincing, but *seeing* it is better. Below, the **same test
images** run through each model — one row per model. The one-epoch model draws
hesitant, often wrong, low-confidence boxes (or none at all); a few epochs
later the boxes are already much more sensible; the fully-trained model is
confident and right. (We lower the confidence threshold to 0.25 here, otherwise
the early model would show nothing at all.)

In [ ]:
test_images = sorted((config.DATASET_DIR / "test" / "images").glob("*.jpg"))
test_labeled = [p for p in test_images if load_boxes(p)]


def compare_models(weights_by_label: dict, images: list, conf: float = 0.25) -> None:
    """One row per model, one column per image, predictions drawn on."""
    fig, axes = plt.subplots(len(weights_by_label), len(images),
                             figsize=(3.4 * len(images), 3.5 * len(weights_by_label)),
                             squeeze=False)
    for row, (label, weights) in enumerate(weights_by_label.items()):
        results = YOLO(str(weights)).predict([str(p) for p in images], conf=conf,
                                             device=DEVICE, verbose=False)
        for col, r in enumerate(results):
            axes[row][col].imshow(r.plot()[..., ::-1])   # .plot() gives BGR -> flip to RGB
            axes[row][col].axis("off")
        axes[row][0].text(-0.06, 0.5, label, transform=axes[row][0].transAxes,
                          rotation=90, va="center", ha="right", fontsize=12)
    fig.tight_layout()
    plt.show()


demo_images = random.Random(7).sample(test_labeled, 4)
compare_models(candidates, demo_images)

## 4. Inference

Inference is the model's day job: image in → detections out, fast. Each
detection is just a **class name**, a **confidence** (how sure the model is,
0–1) and a **bounding box** — everything the game does is built on those three
things.

From here on we use the fully-trained model — the same one that gets exported
to NCNN and copied to the Raspberry Pi.

In [ ]:
BEST_WEIGHTS = PRETRAINED_RUN / "weights" / "best.pt"
if not BEST_WEIGHTS.exists():
    BEST_WEIGHTS = WORKSHOP_RUN / "weights" / "best.pt"   # fallback: our short training
best_model = YOLO(str(BEST_WEIGHTS))

picks = random.Random(21).sample(test_labeled, 8)
results = best_model.predict([str(p) for p in picks], conf=config.CONF_THRESHOLD,
                             device=DEVICE, verbose=False)

fig, axes = plt.subplots(2, 4, figsize=(14.5, 7))
for ax, r in zip(axes.flat, results):
    ax.imshow(r.plot()[..., ::-1])
    ax.axis("off")
fig.tight_layout()
plt.show()

Those drawings are for humans. What the *code* receives is much simpler — a
handful of numbers per detection:

In [ ]:
r = results[0]
print(f"image: {Path(r.path).name}   ({len(r.boxes)} detection(s))")
print()
for box in r.boxes:
    name = r.names[int(box.cls)]
    conf = float(box.conf)
    x0, y0, x1, y1 = (round(v) for v in box.xyxy[0].tolist())
    print(f"  {name:9s}  confidence {conf:.2f}   box ({x0}, {y0}) -> ({x1}, {y1})")

print()
print("That's all the game needs: play.py takes the most confident detection,")
print("turns its class name into a move, and applies the rules from game_logic.py.")

### Play a round against a photo

`game_logic.py` holds the pure rules of the game (no ML in it at all). Chain it
to the detector and we have the whole game in a dozen lines — this is exactly
what `play.py` does on the Raspberry Pi, with a live camera instead of a photo.

**Re-run this cell to play another round!**

In [ ]:
photo = random.choice(test_labeled)                     # a new photo every run
r = best_model.predict(str(photo), conf=config.CONF_THRESHOLD,
                       device=DEVICE, verbose=False)[0]

top = max(r.boxes, key=lambda b: float(b.conf), default=None)  # most confident hand
player = game_logic.detection_to_move(r.names[int(top.cls)] if top is not None else None)
computer = game_logic.random_move()

if player is None:
    print("error — no hand detected confidently enough; run the cell again!")
else:
    outcome = game_logic.decide_winner(player, computer)
    verdict = {"win": "the photo wins! 🏆", "lose": "the computer wins! 🤖",
               "tie": "it's a tie! 🤝"}[outcome]
    print(f"photo plays {player!r} — computer plays {computer!r} → {verdict}")

plt.figure(figsize=(5, 5))
plt.imshow(r.plot()[..., ::-1])
plt.axis("off")
plt.show()

## Where to go from here

* **Export for the Pi** — `uv run export.py` converts `best.pt` to NCNN, ~5×
  faster on the Pi's CPU.
* **Play for real** — copy the export to the Raspberry Pi and run
  `uv run play.py` (README §6 has the exact steps).
* **See what the model sees** — `uv run detect_live.py` streams the camera with
  live detections to your browser; great for debugging lighting and distance.

Things to try in this notebook:

* Train a couple more epochs (`EPOCHS = 5`) — how much closer do you get to the
  60-epoch model?
* Drop the confidence threshold in §4 to `0.25` — more detections but more
  mistakes: that's the precision/recall trade-off, live.
* §1 showed Rock has ~40% more examples than the other classes. Run
  `uv run evaluate.py` and check the per-class scores — can you see the effect?